In [1]:
import os
import shutil
import random

# --- Configuration ---
source_img_dir = "Annotated_Images/train/images"
source_lbl_dir = "Annotated_Images/train/labels" 
base_out_dir = "YOLO_Dataset"

# Splitting ratios
train_ratio, val_ratio, test_ratio = 0.7, 0.2, 0.1

# Create new directory structure
dirs_to_make = [
    f"{base_out_dir}/images/train", f"{base_out_dir}/labels/train",
    f"{base_out_dir}/images/val", f"{base_out_dir}/labels/val",
    f"{base_out_dir}/images/test", f"{base_out_dir}/labels/test"
]
for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)

# Get all images and labels
images = [f for f in os.listdir(source_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
data_pairs = []

for img_name in images:
    lbl_name = os.path.splitext(img_name)[0] + ".txt"
    if os.path.exists(os.path.join(source_lbl_dir, lbl_name)):
        data_pairs.append((img_name, lbl_name))

print(f"Found {len(data_pairs)} matching image-label pairs.")

# Shuffle and split
random.seed(42)
random.shuffle(data_pairs)

total = len(data_pairs)
train_end = int(total * train_ratio)
val_end = train_end + int(total * val_ratio)

train_pairs = data_pairs[:train_end]
val_pairs = data_pairs[train_end:val_end]
test_pairs = data_pairs[val_end:]

def copy_files(pairs, split_name):
    for img_name, lbl_name in pairs:
        shutil.copy(os.path.join(source_img_dir, img_name), os.path.join(base_out_dir, f"images/{split_name}", img_name))
        shutil.copy(os.path.join(source_lbl_dir, lbl_name), os.path.join(base_out_dir, f"labels/{split_name}", lbl_name))
    print(f"Copied {len(pairs)} files to {split_name}.")

copy_files(train_pairs, "train")
copy_files(val_pairs, "val")
copy_files(test_pairs, "test")

Found 198 matching image-label pairs.
Copied 138 files to train.
Copied 39 files to val.
Copied 21 files to test.


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import albumentations as A
import cv2
import os

# --- Configuration ---
train_img_dir = "YOLO_Dataset/images/train"
train_lbl_dir = "YOLO_Dataset/labels/train"
augment_multiplier = 2 # Creates 2 augmented copies per original image

# Define Albumentations Pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.MotionBlur(p=0.3),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

images = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
total_augmented = 0

for img_name in images:
    img_path = os.path.join(train_img_dir, img_name)
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(img_name)[0] + ".txt")
    
    # Read image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Read labels
    bboxes = []
    class_labels = []
    with open(lbl_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) == 5:
                class_labels.append(int(parts[0]))
                bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])
                
    # Generate augmented copies
    for i in range(augment_multiplier):
        try:
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_labels = augmented['class_labels']
            
            # Save augmented image
            aug_img_name = f"aug_{i}_{img_name}"
            aug_img_path = os.path.join(train_img_dir, aug_img_name)
            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            
            # Save augmented labels
            aug_lbl_name = os.path.splitext(aug_img_name)[0] + ".txt"
            aug_lbl_path = os.path.join(train_lbl_dir, aug_lbl_name)
            with open(aug_lbl_path, 'w') as f:
                for bbox, label in zip(aug_bboxes, aug_labels):
                    f.write(f"{label} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            total_augmented += 1
        except Exception as e:
            # ShiftScaleRotate can sometimes push bboxes out of bounds entirely
            pass

print(f"Augmentation complete. Generated {total_augmented} new training images.")